<a href="https://colab.research.google.com/github/AlishbaSoftTeche/unstructuredata_tfidf_repo/blob/main/TF-IDF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/mateenyaqoob/unstructuredata_tfidf_repo.git
%cd unstructuredata_tfidf_repo
!pip install -q -r requirements.txt

Cloning into 'unstructuredata_tfidf_repo'...
remote: Enumerating objects: 84, done.
remote: Counting objects: 100% (84/84), done.
remote: Compressing objects: 100% (53/53), done.
remote: Total 84 (delta 28), reused 59 (delta 19), pack-reused 0 (from 0)
Receiving objects: 100% (84/84), 35.77 KiB | 1.23 MiB/s, done.
Resolving deltas: 100% (28/28), done.
/content/unstructuredata_tfidf_repo
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 947.5/947.5 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.2/60.2 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 55.7 MB/s eta 0:00:00


In [2]:
!cat src/load_data.py

"""Utilities for loading the classroom dataset."""
from typing import List, Tuple

from data.documents import TITLES, DOCUMENTS


def load_documents() -> Tuple[List[str], List[str]]:
    """Return (titles, documents)."""
    return TITLES, DOCUMENTS


if __name__ == "__main__":
    titles, docs = load_documents()
    print(f"Loaded {len(docs)} documents.")
    for t, d in zip(titles, docs):
        print(f"- {t}: {len(d.split())} words")


In [3]:
import sys
sys.path.append("/content/unstructuredata_tfidf_repo")
import numpy as np
import pandas as pd
from src.load_data import load_documents

result = load_documents()
if isinstance(result, dict):
    titles, docs = list(result.keys()), list(result.values())
else:
    titles, docs = result

print("Documents loaded:", len(docs))
for t, d in zip(titles, docs):
    print(f"  {t}: {len(d.split())} words")

Documents loaded: 10
  AI in Healthcare: 85 words
  Climate Change: 83 words
  Basketball Strategy: 85 words
  Italian Pasta: 91 words
  Space Exploration: 86 words
  Cybersecurity: 88 words
  Music Theory: 87 words
  Gardening Vegetables: 89 words
  Economics Inflation: 89 words
  Public Health: 91 words


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(stop_words="english")
X = vectorizer.fit_transform(docs)
print("Shape:", X.shape)
print("Vocabulary size:", len(vectorizer.vocabulary_))

Shape: (10, 537)
Vocabulary size: 537


In [5]:
df = pd.DataFrame(X.toarray(), index=titles, columns=vectorizer.get_feature_names_out())
df.iloc[:3, :10].round(3)

,ability,accelerate,access,accountability,accuracy,accurate,actions,activity,adaptation,add
AI in Healthcare,0.00,0.0,0.0,0.125,0.125,0.000,0.000,0.0,0.000,0.0
Climate Change,0.00,0.0,0.0,0.000,0.000,0.125,0.125,0.0,0.125,0.0
Basketball Strategy,0.13,0.0,0.0,0.000,0.000,0.000,0.000,0.0,0.000,0.0


In [6]:
for title in df.index:
    print(f"{title}:")
    for term, score in df.loc[title].nlargest(5).items():
        print(f"   {term:<20} {score:.3f}")
    print()

AI in Healthcare:
   ai                   0.374
   healthcare           0.249
   use                  0.165
   accountability       0.125
   accuracy             0.125

Climate Change:
   change               0.250
   emissions            0.250
   climate              0.212
   energy               0.212
   accurate             0.125

Basketball Strategy:
   ability              0.130
   analytics            0.130
   basketball           0.130
   breaks               0.130
   chemistry            0.130

Italian Pasta:
   pasta                0.443
   sauces               0.332
   sauce                0.222
   different            0.188
   add                  0.111

Space Exploration:
   space                0.349
   mars                 0.233
   missions             0.233
   accelerate           0.116
   advanced             0.116

Cybersecurity:
   cybersecurity        0.249
   systems              0.212
   data                 0.165
   attackers            0.125
   attacks           

In [9]:
from sklearn.metrics.pairwise import cosine_similarity

def rank_documents(query):
    q_vec = vectorizer.transform([query])
    scores = cosine_similarity(q_vec, X).ravel()
    order = np.argsort(scores)[::-1]
    return [(titles[i], scores[i]) for i in order]

In [8]:
queries = [
    "renewable energy and climate policy",
    "password security and phishing",
    "healthy sleep exercise and diet",
]
for q in queries:
    print(f"Query: {q}")
    for rank, (title, score) in enumerate(rank_documents(q), 1):
        print(f"  {rank}. {title:<30} {score:.3f}")
    print()

Query: renewable energy and climate policy
  1. Climate Change                 0.323
  2. Economics Inflation            0.095
  3. Gardening Vegetables           0.048
  4. Public Health                  0.000
  5. Cybersecurity                  0.000
  6. Music Theory                   0.000
  7. Space Exploration              0.000
  8. Italian Pasta                  0.000
  9. Basketball Strategy            0.000
  10. AI in Healthcare               0.000

Query: password security and phishing
  1. Cybersecurity                  0.176
  2. Public Health                  0.000
  3. Economics Inflation            0.000
  4. Gardening Vegetables           0.000
  5. Music Theory                   0.000
  6. Space Exploration              0.000
  7. Italian Pasta                  0.000
  8. Basketball Strategy            0.000
  9. Climate Change                 0.000
  10. AI in Healthcare               0.000

Query: healthy sleep exercise and diet
  1. Public Health                  

In [10]:
vec2 = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), max_df=0.8, min_df=2)
X2 = vec2.fit_transform(docs)
print("Default vocabulary:", len(vectorizer.vocabulary_))
print("Bigram vocabulary: ", len(vec2.vocabulary_))

Default vocabulary: 537
Bigram vocabulary:  69


In [11]:
df2 = pd.DataFrame(X2.toarray(), index=titles, columns=vec2.get_feature_names_out())
for title in df2.index:
    print(title)
    print("  default:", ", ".join(df.loc[title].nlargest(5).index))
    print("  bigrams:", ", ".join(df2.loc[title].nlargest(5).index))
    print()

AI in Healthcare
  default: ai, healthcare, use, accountability, accuracy
  bigrams: use, careful, clean, disease, improve

Climate Change
  default: change, emissions, climate, energy, accurate
  bigrams: climate, energy, economic, economic growth, efficiency

Basketball Strategy
  default: ability, analytics, basketball, breaks, chemistry
  bigrams: adjust, based, communication, decisions, defenses

Italian Pasta
  default: pasta, sauces, sauce, different, add
  bigrams: different, flavor, improve, long, quality

Space Exploration
  default: space, mars, missions, accelerate, advanced
  bigrams: international, medical, health, data, water

Cybersecurity
  default: cybersecurity, systems, data, attackers, attacks
  bigrams: systems, data, defenses, essential, intelligence

Music Theory
  default: music, musicians, practice, allows, artists
  bigrams: practice, communication, different, improves, live

Gardening Vegetables
  default: soil, pests, year, improves, adds
  bigrams: improve

In [12]:
!pytest -q

...                                                                      [100%]
3 passed in 2.45s
